# Lesson 90 · Conversational RAG and a tool-using agent (ReAct) — a machine that remembers the conversation, cites its sources, and gets things done itself

**Goal**: conversational RAG + a ReAct agent written from scratch (Thought → Action → Observation).

> Where tool use draws the line: the agent decides whether to act; RAG only stuffs documents into the context.

🔗 **Aurora connection**: this lesson implements the whole conversation engine inside the notebook — `podcast_qa()` is the design for the core entry point of the planned `aurora/podcast_intelligence/` (a planned module, not created yet); once it lands, L93/L94's evaluation pipeline and the final demo will call it directly.

← **Previous lesson** · [L89 · A complete RAG pipeline](L89_rag_pipeline.ipynb)

> Last lesson covered **a complete RAG pipeline**: document chunking + TF-IDF retrieval + prompt construction + generation.
> This lesson explores **conversational RAG**, and on top of it writes a **ReAct tool-using agent** from scratch (the Thought → Action → Observation loop).

## Have you ever chatted with ChatGPT for more than 10 turns, only for it to forget what you said at the start?

These are two big pain points of LLMs:

1. **Limited memory**: the context window has a length limit, and the longer the conversation, the easier it is to "forget things"
2. **Knowledge cutoff**: the LLM doesn't know your podcast's content; it only knows the world up to its training-data cutoff

**Conversational RAG solves both at once**:

```
your question → [sliding-window history truncation] → keep only the last 5 turns
               ↓
         [TF-IDF retrieval] → fish the 3 most relevant podcast passages out of the knowledge base
               ↓
         [prompt assembly] → join history + retrieved results + question into one structured prompt
               ↓
         [LLM generation] → answer only from the context (no making things up)
               ↓
         answer + source labels (which episode, which minute)
```

**Three engineering building blocks**:
- `build_prompt(context_chunks, history, question)` — assembles the full prompt
- `format_sources(source_chunks)` — formats the source attribution
- `truncate_history(history, max_turns=5)` — a sliding window that keeps the token count from blowing up

**Main function**: `podcast_qa(question, rag_index, chunks, embed_fn, llm, history)` — a 5-step pipeline that strings these three together.

## Key intuition: the LLM is the cook, RAG is the recipe

Without RAG, the LLM is like a cook working purely "by feel" — it might turn out a brilliant dish, or it might invent a recipe that never existed (hallucination).

With RAG, you hand the cook a **recipe pulled out on the spot** (the retrieved chunks). They have to follow it and can't throw in extra ingredients — which is why the prompt says "answer only from the content above; if the context doesn't contain the answer, say clearly that you can't answer".

**Why put the context before the history?**
The model reads the "factual text" first and the "conversation flow" second, which stops the history from interfering with the retrieved results — otherwise the model might treat things said earlier in the chat as new "facts".

**The engineering value of source attribution**:
- Users can click a timestamp to jump back into the podcast and verify
- In automatic evaluation, Precision@k = how many of the k retrieved chunks contain the correct answer

In [ ]:
# ── aurora.llm import (TF-IDF sparse retrieval) ──────────────────────────────
#
# L89 implemented build_tfidf / cosine_retrieve (sparse vectors of term counts × inverse document frequency).
# On top of that, this lesson provides the compatible interface build_rag_index / retrieve, and shows how to
# extend it to a dense embedding path — see the embed_fn parameter in the stub implementation below.
#
# Sparse vs dense retrieval:
#   TF-IDF (L89/L90 aurora.llm): term counting, no model needed, fast, useless for OOV words
#   Dense embedding (L90 stub): semantic similarity, needs an embedding model, generalizes better but is heavier
try:
    from aurora.llm import build_tfidf, cosine_retrieve

    def build_rag_index(chunks, embed_fn=None):
        """Build the retrieval index with aurora.llm TF-IDF.

        Returns a (tfidf_matrix, vocab, chunks) triple for retrieve() to use.
        The embed_fn parameter is kept but ignored on the TF-IDF path.
        """
        texts = [c["text"] if isinstance(c, dict) else c for c in chunks]
        matrix, vocab = build_tfidf(texts)
        return (matrix, vocab, chunks)

    def retrieve(query, rag_index, chunks, embed_fn=None, top_k=3):
        """TF-IDF cosine-similarity retrieval; returns a list of the top_k most relevant chunk dicts."""
        matrix, vocab, orig_chunks = rag_index
        texts = [c["text"] if isinstance(c, dict) else c for c in orig_chunks]
        results = cosine_retrieve(query, matrix, vocab, texts, top_k=top_k)
        # Map the retrieved text back to the original chunk dicts (keeping the source field)
        text_to_chunk = {}
        for ch in orig_chunks:
            t = ch["text"] if isinstance(ch, dict) else ch
            text_to_chunk[t] = ch
        return [text_to_chunk.get(doc, {"text": doc, "source": ""})
                for doc, _ in results]

    print("✅ aurora.llm imported (TF-IDF retrieval path)")
except ImportError:
    import numpy as np

    def _cosine(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    def build_rag_index(chunks, embed_fn=None):
        """Dense-embedding stub index (the fallback when aurora.llm isn't available).

        Returns the embedding matrix as an ndarray of shape (n_chunks, dim).
        Note: if embed_fn is None, random vectors are used (for demonstration only).
        """
        import numpy as np
        if embed_fn is None:
            rng = np.random.default_rng(0)
            return rng.standard_normal((len(chunks), 8)).astype(np.float32)
        vecs = np.array([
            embed_fn(c["text"] if isinstance(c, dict) else c) for c in chunks
        ], dtype=np.float32)
        return vecs

    def retrieve(query, rag_index, chunks, embed_fn=None, top_k=3):
        """Dense-embedding cosine retrieval (stub version)."""
        import numpy as np
        vecs = rag_index
        if embed_fn is None:
            return chunks[:top_k]
        q_vec = embed_fn(query)
        scores = [_cosine(q_vec, v) for v in vecs]
        idxs = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [chunks[i] for i in idxs]

    print("⚠️  Using the dense-embedding stub (aurora.llm not found; run make install)")

## 1. The RAG prompt format

A RAG prompt is built from four layers joined together, and the order matters:

```
[SYSTEM]
You are a podcast assistant. Answer only from the context provided; do not make things up.

[CONTEXT]
--- Passage 1 (source: episode_42.txt, t=00:12:30) ---
"John played lead guitar on that track..."

--- Passage 2 (source: episode_42.txt, t=00:15:02) ---
"The band lineup included Sarah on bass..."

[HISTORY]
User: Who produced the album?
Assistant: The album was produced by Rick Rubin (source: episode_42.txt, t=00:08:10).

[QUESTION]
Who played guitar in this episode?

Answer only from the content above. If the context does not contain the answer, say clearly "I can't answer that from the available content".
```

Key design decisions:
- `[CONTEXT]` comes before `[HISTORY]` — the model reads the facts first and the conversation flow second, which reduces the history's interference with the retrieved results
- "Answer only from the content above" is a hard constraint that stops the model mixing in its pretraining memory
- Every passage is labeled with its source, ready for source attribution later

In [ ]:
def build_prompt(context_chunks, history, question, system_prompt=None):
    """
    Join the retrieved passages + conversation history + current question into one prompt string.

    Parameters
    ----------
    context_chunks : list[dict]
        Each element has 'text' and 'source' keys.
    history : list[dict]
        Each element has 'role' ('user'/'assistant') and 'content' keys.
    question : str
        The current user question.
    system_prompt : str | None
        If None, the default system prompt is used.

    Returns
    -------
    str
        The assembled prompt.
    """
    if system_prompt is None:
        system_prompt = (
            "You are a podcast assistant. Answer only from the context provided; do not make things up.\n"
            "If the context does not contain the answer, say clearly \"I can't answer that from the available content\"."
        )

    parts = [f"[SYSTEM]\n{system_prompt}\n"]

    # Context passages
    parts.append("[CONTEXT]")
    for i, chunk in enumerate(context_chunks, 1):
        src = chunk.get("source", "unknown source")
        text = chunk.get("text", "")
        parts.append(f"--- Passage {i} (source: {src}) ---\n{text}")
    parts.append("")

    # Conversation history
    if history:
        parts.append("[HISTORY]")
        for turn in history:
            role = "User" if turn["role"] == "user" else "Assistant"
            parts.append(f"{role}: {turn['content']}")
        parts.append("")

    # Current question
    parts.append(f"[QUESTION]\n{question}\n\nAnswer only from the content above.")

    return "\n".join(parts)


# Demo
demo_chunks = [
    {"text": "John played lead guitar on that track.", "source": "ep42.txt t=00:12:30"},
    {"text": "Sarah was on bass for the whole session.", "source": "ep42.txt t=00:15:02"},
]
demo_history = [{"role": "user", "content": "Who produced the album?"},
                {"role": "assistant", "content": "Rick Rubin (source: ep42.txt t=00:08:10)."}]
prompt = build_prompt(demo_chunks, demo_history, "Who played guitar?")
print(prompt)

## 2. Source attribution

Every retrieval should record which document, and where in it, the passage was found. The format:

```
Source: episode_42.txt, t=00:12:30
```

How it's implemented: every chunk dict returned by `retrieve()` carries a `source` field; `podcast_qa()` returns the list of those sources to the caller, and a front end or evaluation script can display them or log them directly.

Why source attribution is worth it:
- Users can verify the answer (click the timestamp to jump back into the podcast)
- During evaluation you can automatically compare the "retrieved sources" with the "labeled answer sources" and compute Precision@k

In [ ]:
def format_sources(source_chunks):
    """
    Format a list of source chunks as a list of readable strings.

    Parameters
    ----------
    source_chunks : list[dict]
        A list of chunks with a 'source' key.

    Returns
    -------
    list[str]
        A list of formatted source strings, deduplicated with order preserved.
    """
    seen = set()
    result = []
    for chunk in source_chunks:
        src = chunk.get("source", "unknown source")
        if src not in seen:
            seen.add(src)
            result.append(f"📄 {src}")
    return result


# Demo
srcs = format_sources(demo_chunks)
for s in srcs:
    print(s)
print("✅ Source formatting works")

## 3. Conversation memory and the sliding window

The longer the conversation history, the more tokens it costs, until it eventually exceeds the LLM's context length. The fix: a **sliding window** — keep only the last `k` turns of the conversation.

```
history = history[-(2*k):]   # each turn has one user + one assistant entry, 2k entries in total
```

A finer-grained approach truncates by token count rather than by turns, but for podcast QA, k=5 turns (≈1000 tokens) is plenty.

Note: truncating the history doesn't affect retrieval quality — retrieval works on the question alone in the embedding space and has nothing to do with the history.

In [ ]:
def truncate_history(history, max_turns=5):
    """
    Keep the last max_turns turns of the conversation (each turn = one user + one assistant entry).

    Parameters
    ----------
    history : list[dict]
        The conversation history; each entry has 'role' and 'content'.
    max_turns : int
        The maximum number of turns to keep.

    Returns
    -------
    list[dict]
        The truncated history.
    """
    max_entries = max_turns * 2   # user + assistant
    return history[-max_entries:] if len(history) > max_entries else history


# Demo: build an 8-turn history and truncate it to 3 turns
long_history = []
for i in range(8):
    long_history.append({"role": "user", "content": f"Question {i+1}"})
    long_history.append({"role": "assistant", "content": f"Answer {i+1}"})

short = truncate_history(long_history, max_turns=3)
print(f"Original history entries: {len(long_history)}")
print(f"After truncation:         {len(short)}")
print(f"Earliest turn kept:       {short[0]['content']}")
print("✅ Sliding window works")

## 4. ✏️ Implement `podcast_qa(question, rag_index, chunks, embed_fn, llm, history=None)`

**Line of reasoning**:
1. `truncate_history(history, max_turns=5)` → trim the history so the token count doesn't blow up
2. `retrieve(question, rag_index, chunks, embed_fn, top_k=3)` → get the 3 most relevant context chunks
3. `build_prompt(context_chunks, history, question)` → assemble the full prompt
4. `llm.generate(prompt)` → call the local LLM to generate the answer string
5. Return `(answer: str, sources: list[str])`, with sources produced by `format_sources(context_chunks)`

**Reference input and output**:

```python
question  = "Who played guitar in this episode?"
# retrieve finds:
#   chunk_0: {"text": "John played lead guitar...", "source": "ep42.txt t=00:12:30"}
#   chunk_1: {"text": "The session featured...",   "source": "ep42.txt t=00:14:55"}
#   chunk_2: {"text": "Sarah on bass...",           "source": "ep42.txt t=00:15:02"}
# llm generates:
answer  = "John played lead guitar in this episode."
sources = ["📄 ep42.txt t=00:12:30", "📄 ep42.txt t=00:14:55", "📄 ep42.txt t=00:15:02"]
```

> 💡 Need a hint? Once you've done the exercise, you can compare with the reference implementation in the `solutions/` folder.


In [ ]:
def podcast_qa(question, rag_index, chunks, embed_fn, llm, history=None):
    """
    The podcast Q&A main function: retrieve relevant passages → assemble the prompt → LLM generates → return the answer and sources.

    Parameters
    ----------
    question  : str          the current user question
    rag_index : np.ndarray   the index produced by build_rag_index (a (matrix, vocab, chunks) triple on the TF-IDF path)
    chunks    : list[dict]   the original list of chunks, each with 'text' and 'source'
    embed_fn  : callable     an embedding function, text → vector
    llm       : object       an LLM object with a .generate(prompt: str) -> str method
    history   : list[dict]   the conversation history, defaults to None (an empty list)

    Returns
    -------
    answer  : str
    sources : list[str]
    """
    if history is None:
        history = []

    # ✏️ TODO step 1: trim history with truncate_history (max_turns=5)
    raise NotImplementedError("TODO: trim history with truncate_history (max_turns=5)")

    # ✏️ TODO step 2: call retrieve to get context_chunks (top_k=3)
    raise NotImplementedError("TODO: call retrieve to get context_chunks (top_k=3)")

    # ✏️ TODO step 3: call build_prompt to assemble the full prompt
    raise NotImplementedError("TODO: call build_prompt to assemble the full prompt")

    # ✏️ TODO step 4: call llm.generate(prompt) to get answer
    raise NotImplementedError("TODO: call llm.generate(prompt) to get answer")

    # ✏️ TODO step 5: call format_sources and return (answer, sources)
    raise NotImplementedError("TODO: call format_sources to build the list of sources")
    return answer, sources

In [ ]:
# ── A minimal stub LLM for unit tests (no real model is called) ──────────────
import numpy as np

class _StubLLM:
    """A fake LLM: echoes back the retrieved passage text from the prompt's [CONTEXT] section, for checking only.

    A real LLM reads the whole prompt and composes an answer; here we use a deterministic stub
    that only extracts the passage text from the [CONTEXT] section and joins it together, ignoring [SYSTEM]/[HISTORY]/[QUESTION].
    That way, "with RAG" the answer is guaranteed to quote the retrieved chunk text, which makes it easy to check that the pipeline is connected.
    """
    def generate(self, prompt):
        context_lines = []
        in_context = False
        for line in prompt.split("\n"):
            stripped = line.strip()
            if stripped.startswith("[CONTEXT]"):
                in_context = True
                continue
            if in_context and stripped.startswith("["):   # reached [HISTORY]/[QUESTION]
                break
            # skip the "--- Passage N (source: ...) ---" separator lines, keep only the body
            if in_context and stripped and not stripped.startswith("---"):
                context_lines.append(stripped)
        if context_lines:
            return " ".join(context_lines)   # use all the retrieved passages together as the answer
        return "Can't tell from the context provided."

def _stub_embed(text_or_chunk):
    """Hash a text into an 8-dimensional vector (for tests only)."""
    text = text_or_chunk["text"] if isinstance(text_or_chunk, dict) else text_or_chunk
    rng = np.random.default_rng(abs(hash(text)) % (2**31))
    return rng.standard_normal(8).astype(np.float32)

# Build the test data
_test_chunks = [
    {"text": "John played lead guitar on that track.", "source": "ep42.txt t=00:12:30"},
    {"text": "Sarah was on bass for the whole session.", "source": "ep42.txt t=00:15:02"},
    {"text": "The producer was Rick Rubin.", "source": "ep42.txt t=00:08:10"},
    {"text": "Drums were handled by Mike.", "source": "ep42.txt t=00:20:00"},
]
_test_index = build_rag_index(_test_chunks, _stub_embed)
_test_llm   = _StubLLM()

try:
    answer, sources = podcast_qa(
        question   = "Who played guitar in this episode?",
        rag_index  = _test_index,
        chunks     = _test_chunks,
        embed_fn   = _stub_embed,
        llm        = _test_llm,
        history    = [],
    )
except (NotImplementedError, TypeError):
    print('⬜ Implement the 5 TODO steps of podcast_qa() first')
else:
    print('Answer:', answer)
    print('Sources:')
    for s in sources:
        print('  ', s)
    assert isinstance(answer, str) and len(answer) > 0, 'answer should be a non-empty string'
    assert isinstance(sources, list) and len(sources) > 0, 'sources should be a non-empty list'
    assert all(s.startswith('📄') for s in sources), 'every source should start with 📄'
    print('\n✅ podcast_qa basic check passed')


## 5. Parameter experiment: answer quality with and without RAG

**Setup**:
- Build an index of 10 simulated podcast episodes (4 chunks per episode, 40 in total)
- Test 5 questions, recording each one:
  - `with_rag=True`: run the full `podcast_qa()` flow
  - `with_rag=False`: feed the question straight to the LLM (no retrieval, no injected context)

**What to expect**:
- `top_k=3`: usually enough; raising it to 5 may bring in noisy chunks and make the answer less focused
- `max_turns=5`: once the history passes 5 turns it gets truncated and answers stay stable; set it to 0 and every turn stands alone, losing the conversation's continuity
- Without RAG: the LLM tends to hallucinate or say "I don't know" about podcast-specific details (names, timestamps, opinions)
- With RAG: the answer quotes the chunk text directly, and the source can be traced

**Key metric**: `Precision@3` (how many of the 3 retrieved results are chunks containing the correct answer)

In [ ]:
import numpy as np

# ── Build a synthetic dataset of 10 episodes × 4 chunks ──────────────────────
np.random.seed(42)

episode_facts = [
    ("Alice", "piano"), ("Bob", "guitar"), ("Carol", "drums"),
    ("Dave", "bass"),   ("Eve", "violin"), ("Frank", "trumpet"),
    ("Grace", "flute"), ("Hank", "cello"), ("Iris", "harp"), ("Jack", "sitar"),
]

synth_chunks = []
for ep_idx, (name, instrument) in enumerate(episode_facts):
    ep_name = f"episode_{ep_idx+1:02d}.txt"
    synth_chunks += [
        {"text": f"{name} played {instrument} in the session.",
         "source": f"{ep_name} t=00:10:{ep_idx*3:02d}"},
        {"text": f"The host interviewed {name} about their career.",
         "source": f"{ep_name} t=00:20:{ep_idx*2:02d}"},
        {"text": f"Audience Q&A focused on {instrument} technique.",
         "source": f"{ep_name} t=00:35:00"},
        {"text": f"Closing remarks from {name}.",
         "source": f"{ep_name} t=00:55:00"},
    ]

synth_index = build_rag_index(synth_chunks, _stub_embed)

# ── 5 test questions (with the correct answers for comparison) ───────────────
test_questions = [
    ("Who plays guitar?",  "Bob"),
    ("Who plays piano?",   "Alice"),
    ("Who plays drums?",   "Carol"),
    ("Who plays violin?",  "Eve"),
    ("Who plays cello?",   "Hank"),
]

# ── The comparison experiment ────────────────────────────────────────────────
class _NoContextLLM:
    """Simulates an LLM without RAG: it ignores the context and just guesses."""
    def generate(self, prompt):
        return "I'm not sure, probably some musician."   # a typical hallucination/refusal

try:
    print(f"{'Question':<30} {'Expected':<10} {'RAG answer':<55} {'No-RAG answer'}")
    print("-" * 120)
    n_hit = 0
    for question, expected in test_questions:
        # with RAG
        ans_rag, srcs = podcast_qa(question, synth_index, synth_chunks, _stub_embed, _test_llm)
        # without RAG (call the LLM directly, skipping retrieval)
        ans_nrag = _NoContextLLM().generate(question)
        hit = expected.lower() in ans_rag.lower()
        n_hit += hit
        mark = "✅" if hit else "❌"
        print(f"{question:<30} {expected:<10} {mark} {ans_rag[:50]:<52} {ans_nrag}")
    print()
    print(f"Hit rate (RAG answer contains the correct name): {n_hit}/{len(test_questions)}")
    print("Observation: with RAG the answer quotes the chunk text directly; without RAG the LLM can't answer podcast-specific details.")
except (NotImplementedError, TypeError):
    print("⬜ Implement podcast_qa() first, then run the comparison experiment")

## ✏️ Closed-book derivation check cell — the conversational RAG data flow

**Rule: close every cell above and work it out from memory.**

**Problem**: for the 5-step pipeline of `podcast_qa()`, fill in each step's input → output type:

| Step | Function | Input | Output |
|---|---|---|---|
| 1 | `truncate_history` | `history: list[dict]` | ___ (list[dict], at most max_turns*2 entries) |
| 2 | `retrieve` | `question: str, rag_index, chunks` | ___ (list[dict], top_k chunks) |
| 3 | `build_prompt` | `context_chunks, history, question` | ___ (str, the full prompt) |
| 4 | `llm.generate` | `prompt: str` | ___ (str, the answer) |
| 5 | `format_sources` | `context_chunks: list[dict]` | ___ (list[str], the list of sources) |

And answer:
1. Why does `truncate_history` have to be called before `build_prompt` (and why doesn't it change what `retrieve` returns)?
2. What data structure does the deduplication logic in `format_sources` use? What's its time complexity?

(Fill in here...)

In [ ]:
# Check: podcast_qa end-to-end integration (if implemented)
try:
    _q = "Who plays guitar?"
    _ans, _srcs = podcast_qa(
        question=_q,
        rag_index=_test_index,
        chunks=_test_chunks,
        embed_fn=_stub_embed,
        llm=_test_llm,
        history=[],
    )
    assert isinstance(_ans, str) and len(_ans) > 0, "answer should be a non-empty string"
    assert isinstance(_srcs, list) and len(_srcs) > 0, "sources should be a non-empty list"
    assert all(isinstance(s, str) for s in _srcs), "every source should be a string"

    # History truncation check
    _long_history = [
        {"role": "user" if i % 2 == 0 else "assistant", "content": f"Turn {i//2+1}"}
        for i in range(20)
    ]
    _trunc = truncate_history(_long_history, max_turns=3)
    assert len(_trunc) == 6, f"3 turns = 6 entries, got {len(_trunc)}"
    assert _trunc[0]["content"] == "Turn 8", f"the first entry should be turn 8, got {_trunc[0]['content']}"

    print("✅ podcast_qa end-to-end check passed")
    print(f"  Answer: {_ans[:50]}")
    print(f"  Sources: {_srcs}")
    print(f"  History truncation: 20 entries → 6 entries (max_turns=3)")
except (NotImplementedError, TypeError):
    print("⚠️ podcast_qa isn't implemented yet; re-run after finishing the 5 TODOs")

## 6. From "citing its sources" to "doing it itself" — a ReAct agent (tool use)

The `podcast_qa` above already knows how to "look things up before answering". But it only knows one road: retrieve → generate.
Ask it something like "who plays guitar — and while you're at it, work out 6×7", which needs both **looking up** and **calculating**, and plain RAG gets stuck —
the knowledge base has no answer for "6×7", and LLMs often slip up at mental arithmetic.

**The agent's idea**: don't let the LLM blurt out the whole answer in one go; instead, like a person working through a problem, **think while using tools**:

```
Thought:     First I need to find out who plays guitar.
Action:      search[who plays guitar]
Observation: Bob played guitar in the session.
Thought:     Good, next I'll work out 6×7.
Action:      calculator[6*7]
Observation: 42
Thought:     Both questions have answers now.
Final Answer: Bob plays guitar; 6×7=42.
```

This is the **ReAct** (Reasoning + Acting) loop: **Thought → Action (call a tool) → Observation (look at the result)**,
round and round, until the LLM spits out `Final Answer:` and stops.

**The key point: who is actually turning this loop?**
- **The LLM only decides "what to do next"**: it reads the current scratchpad once and produces one `Thought:` line + one `Action: tool_name[input]`, or announces `Final Answer:`. It does **not** actually execute anything.
- **The loop body (code we write ourselves) is the real executor**: it parses out `Action: tool_name[input]`, calls that real Python function, pastes the return value back onto the scratchpad as `Observation:`, and feeds it to the LLM for the next round.

**Why not use langchain?** Aurora's iron rule is **no API wrappers** — take the ReAct loop apart and it's nothing more than "regex parsing + function dispatch + string concatenation", which twenty lines can cover completely. A framework would only hide the very logic you most need to understand inside a black box. Write it once yourself and you'll really know what an agent is doing.

**The two tools in this section (both reuse existing course code — CPU, offline, reproducible)**:
- `calculator(expr)`: a safe expression evaluator built from scratch, never touches the network.
- `search(query)`: wraps **the from-scratch TF-IDF retriever from L89/L90** directly as a tool — retrieval itself is one of the agent's "senses".


In [ ]:
import re

# ── Tool 1: calculator — a safe evaluator built from scratch, no external libraries ──
def calculator(expr):
    """A safe expression evaluator that only allows digits and + - * / ( ) . and spaces."""
    allowed = set("0123456789+-*/(). ")
    if not set(expr) <= allowed:
        return "Error: the expression contains illegal characters"
    try:
        # __builtins__ emptied out to rule out arbitrary code execution (not an API wrapper — our own sandbox)
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

# ── Tool 2: search — reuses the from-scratch TF-IDF retriever from L89/L90 ──────
def search(query):
    """Wrap TF-IDF retrieval as an agent tool; returns the single most relevant sentence."""
    hits = retrieve(query, synth_index, synth_chunks, _stub_embed, top_k=1)
    return hits[0]["text"] if hits else "No results"

TOOLS = {"calculator": calculator, "search": search}

# ── A deterministic ReAct planner: a scripted "fake LLM", fully reproducible, zero network ──
class ScriptedReActLLM:
    """Plays the LLM in ReAct: counts how many Observations are already on the scratchpad and decides the next step from that.

    A real LLM plans through language understanding; here a deterministic script that "counts Observations" stands in,
    so the classroom result is exactly the same every time. The strings it produces strictly follow the ReAct text protocol:
      · intermediate step: one `Thought:` line + one `Action: tool_name[input]` line
      · finish:            `Final Answer: ...`
    """
    def generate(self, prompt):
        obs = re.findall(r"Observation:\s*(.*)", prompt)
        n = len(obs)
        if n == 0:
            return ("Thought: First I need to look up who plays guitar.\n"
                    "Action: search[who plays guitar]")
        if n == 1:
            return ("Thought: Now I know the musician; next, work out 6*7.\n"
                    "Action: calculator[6*7]")
        # Two observations in hand: put the answer together
        return f"Final Answer: on guitar: \"{obs[0]}\"; 6*7={obs[1]}."

# First unit-test the two tools and the planner's first step on their own (the scratchpad has no Observation yet)
print("calculator('6*7')      ->", calculator("6*7"))
print("search('who plays guitar') ->", search("who plays guitar"))
print("LLM step 1:\n" + ScriptedReActLLM().generate("Question: demo\n"))


## 7. ✏️ Implement the ReAct loop `run_agent(question, llm, tools, max_steps=5)`

Your turn to get this "think — act — look" loop turning yourself. Each round, the loop body does 5 things:

1. Call `llm.generate(scratchpad)` to get this step's text `step`;
2. Append `step` to `scratchpad` (and record it in `trace`);
3. If `Final Answer:` appears in `step`, take the text after it as `answer` and return `(answer, trace)`;
4. Otherwise parse `Action: tool_name[input]` with a regex, look up the matching function in `tools` and call it to get `observation`;
5. Append `Observation: {observation}` to `scratchpad` (and `trace`) and go on to the next round.

If there's still no `Final Answer:` after `max_steps`, raise an error (to prevent an infinite loop).


In [ ]:
def run_agent(question, llm, tools, max_steps=5):
    """Run the ReAct loop: Thought → Action → Observation → ... → Final Answer.

    Parameters
    ----------
    question  : str              the user question
    llm       : object           a planner with .generate(prompt) -> str
    tools     : dict[str, func]  tool name → callable (takes one str, returns one str)
    max_steps : int              the maximum number of rounds (prevents infinite loops)

    Returns
    -------
    (answer: str, trace: list[str])
        answer is the text after `Final Answer:`; trace is the record of every step's text (including Observations).
    """
    scratchpad = f"Question: {question}\n"   # the scratchpad: Thought/Action/Observation keep getting appended
    trace = []
    for _ in range(max_steps):
        # TODO 1: step = llm.generate(scratchpad)
        # TODO 2: append step to scratchpad and trace
        # TODO 3: if "Final Answer:" is in step, take the text after it as answer, return (answer, trace)
        # TODO 4: otherwise parse the tool name and input with re.search(r"Action:\s*(\w+)\[(.*?)\]", step),
        #          look up the function in tools and call it to get observation
        # TODO 5: append f"Observation: {observation}" to scratchpad and trace, then go on to the next round
        raise NotImplementedError(
            "implement one step of the ReAct loop: parse the Action → call the tool → append the Observation")
    raise RuntimeError(f"went past max_steps={max_steps} without reaching a Final Answer")


In [ ]:
# ✅ Check: the ReAct agent should reach the right answer in a bounded number of steps using both tools
try:
    _answer, _trace = run_agent(
        "Who plays guitar? And while you're at it, work out 6 times 7.",
        ScriptedReActLLM(),
        TOOLS,
        max_steps=5,
    )
except (NotImplementedError, TypeError):
    print("⬜ Implement the ReAct loop in run_agent() first")
else:
    print("=== Agent trace (scratchpad) ===")
    for _line in _trace:
        print(_line)
    print("\nFinal answer:", _answer)

    _n_obs = sum(1 for l in _trace if l.startswith("Observation:"))
    assert "Bob" in _answer, "the search tool should find that Bob plays guitar"
    assert "42" in _answer, "the calculator tool should work out 6*7=42"
    assert _n_obs == 2, f"the tools should be called exactly twice (two Observations), got {_n_obs}"
    assert len(_trace) <= 2 * 5, "should converge in a bounded number of steps, not run up against max_steps"
    print("\n✅ The ReAct agent used the search + calculator tools and reached the right answer in a bounded number of steps")


## Wrap-up

This lesson implemented `podcast_qa(question, rag_index, chunks, embed_fn, llm, history)` — it joins the retrieved context chunks into a structured prompt and outputs an `(answer: str, sources: list[str])` pair, so every answer can be traced back to a specific podcast timestamp. The three helpers `build_prompt`, `truncate_history` and `format_sources` are implemented in full in this notebook and form the skeleton blueprint for the planned `aurora/podcast_intelligence/qa.py` (a planned module, not created yet).

On top of that, this lesson also wrote a **ReAct tool-using agent** from scratch: `run_agent()` repeatedly runs the **Thought → Action (tool) → Observation** loop until the LLM gives a `Final Answer:`. Two tools — `calculator`, built from scratch, and `search`, which reuses the TF-IDF retriever — let the agent both calculate and look things up, all on CPU, offline and reproducible, without any agent framework. Next lesson: **L91**.

In [ ]:
# ✏️ Self-review for this lesson
l90_review = {
    "rag_pipeline_intuition":    None,  # understand conversational RAG = sliding-window history + retrieval + prompt assembly + LLM generation? True/False
    "build_prompt_understood":   None,  # understand the four-layer prompt structure (SYSTEM/CONTEXT/HISTORY/QUESTION)? True/False
    "podcast_qa_impl":           None,  # podcast_qa() 5 steps are correct and the end-to-end checker passes? True/False
    "source_attribution":        None,  # format_sources deduplicates while keeping order, and I understand Precision@k? True/False
    "whiteboard_passed":         None,  # cleared the whiteboard challenge of filling in the 5-step data-flow table? True/False
    "react_agent_impl":          None,  # run_agent() ReAct loop is correct, and the agent reaches the right answer with both tools? True/False
}

unfilled = [k for k, v in l90_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l90_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L90 all clear! On to L91: attention illustrated')

---

→ **Next lesson** · [L91 · Attention illustrated](L91_visual_llm.ipynb)

> Next lesson covers **attention illustrated**: heatmaps of single-head self-attention weights, and a visualization of LoRA's low-rank structure.